# Navegar e verificar lineage

POC DEMO — contrato PROVISÓRIO / PENDENTE DE HOMOLOGAÇÃO. Execute em kernel novo. Credenciais somente por ambiente.


Configuração tipada. Pré-requisitos: Docker saudável, seed DEMO, token por ambiente. Entradas: parâmetros abaixo; saídas: tabelas, IDs e relatório JSON/Markdown. Escritas exigem confirmação EXECUTE_DEMO e chave estável; repetir chave igual recupera o mesmo job. Espere READY; erros de dependência exigem verificar docker compose logs.

In [1]:
import os, json
import pandas as pd
from IPython.display import display, Markdown
from src.workbench.journey_config import Settings
cfg=Settings.load()
from src.workbench.notebook_config import Parameters
from src.workbench.operations_sdk import call, report, await_job
params=Parameters.load()
display(params.model_dump(mode="json"))
display({"state":"READY", "write_enabled":params.execute_write, "tracking":"remote Docker service", "browser_ui":cfg.ui})
evidence={}
NOTEBOOK_ID="08_auditoria_lineage"


{'mode': 'READ_EXISTING',
 'proposal_id': '712b11cd-a96a-4bb9-83bb-691724ceae88',
 'snapshot_id': None,
 'model_name': 'planilha-turmas-demo',
 'model_version': '',
 'registry_prefix': '',
 'registry_page_token': '',
 'campaign_id': None,
 'approval_id': None,
 'execute_write': False,
 'confirmation': '',
 'request_key': '',
 'requester': 'scientist-demo',
 'algorithm': 'ridge',
 'alpha': 1.0,
 'capacity': 40,
 'model_count': 30,
 'parallel_jobs': 2,
 'campaign_action': 'observe',
 'governance_action': 'read',
 'review_action': 'APPROVE',
 'reason': '',
 'lineage_kind': 'proposal',
 'external_id': '',
 'reset_mode': 'preview',
 'reinitialize': 'demo-ready',
 'reset_plan_id': '8a186e1f-3eb9-4819-b6ec-f99ff331853e'}

{'state': 'READY',
 'write_enabled': False,
 'tracking': 'remote Docker service',
 'browser_ui': 'http://127.0.0.1:5000'}

1. Escolha proposta, run, modelo+versão, snapshot ou campanha; paginação limita o grafo retornado.

In [2]:
from src.workbench.lineage_review import inspect_graph
external_id=params.external_id or (str(params.proposal_id) if params.proposal_id else "")
if not external_id: raise ValueError("Informe external_id ou proposal_id")
evidence=inspect_graph(cfg,params.lineage_kind,external_id,params.model_version)
display(pd.DataFrame(evidence["nodes"]));display(pd.DataFrame(evidence["edges"]))


,entity_id,entity_type,external_id,environment,domain,created_at_utc,attributes
0,f2088b18-0a06-4413-872b-3573b404aa23,campaign,28d5c5e0-e21d-458c-8e0a-3aca7b4acc31,poc,planilha-turmas,2026-11-04T18:58:35.755561+00:00,{}
1,bd8c45c2-c137-45e5-92d1-d89be5b1f94b,campaign,39f8c7aa-0e6a-463c-aa77-3748f6d118d5,poc,planilha-turmas,2026-11-04T20:12:06.733851+00:00,{}
2,84922b87-59fb-48aa-9935-5be737802ae1,campaign,4e413355-a21c-4c13-9311-0320857de96c,poc,planilha-turmas,2026-11-04T18:13:16.517767+00:00,{}
3,b8dce745-f51c-4632-b2b8-21514dde796f,campaign,5f1ac7f1-ce17-4ec9-bdaf-53f84853b673,poc,planilha-turmas,2026-11-04T19:24:42.416011+00:00,{}
4,724552da-7f4b-4d8c-870e-f978bd2a8a06,campaign,7d156255-ee18-4fb6-9c09-a7626dd5e1da,poc,planilha-turmas,2026-11-04T19:24:30.102106+00:00,{}
...,...,...,...,...,...,...,...
56,dde8f698-7130-47cf-b1a3-f653b683667a,run,c78c2971d2d246efb3fcacc5cdf4cd71,poc,planilha-turmas,2026-11-04T19:24:42.416011+00:00,{}
57,ef484b18-2020-4695-b569-d11f16e6b566,run,e400ad37fd404c7ab55e7d50b4b16282,poc,planilha-turmas,2026-11-04T19:24:30.102106+00:00,{}
58,7834b450-1f6f-485c-90c0-a3b97c0f0584,run,e5eb660aa44b4542b7281105c34613da,poc,planilha-turmas,2026-11-04T15:41:06.535205+00:00,{}
59,f52962cd-0110-4425-ae7f-1e28570bf565,run,f543e6c592254507a10b7e9f1381c94c,poc,planilha-turmas,2026-11-04T20:12:06.733851+00:00,{}


,edge_id,parent_id,child_id,relation_type,created_at_utc
0,042f4a44-63da-4576-a04f-50f8d65ef159,70ae9d3c-2b48-4bd4-a565-91d18ec39d42,f52962cd-0110-4425-ae7f-1e28570bf565,TRAINED_ON,2026-11-04T20:12:06.733851+00:00
1,0eba76bd-c6e8-4c1a-ba81-c960f9cc3efa,24c9141e-586c-4e2b-b035-8761100b3daf,3c910bc8-5bf7-4818-86d4-2278a9e3ae77,CONTRIBUTED_TO,2026-11-04T19:29:06.312865+00:00
2,0f6f3fff-9d04-4b5a-9a50-ff4b53fa0a03,d1fa2274-3e4f-4718-8249-b147bfeccd1b,dd0ee492-298a-4155-b597-362af976060b,PREDICTED_WITH,2026-11-04T19:24:44.231085+00:00
3,11cd13ea-1289-4d1c-baa8-0070eddfc1db,d49fff81-3165-4c45-8a39-055e00d10f39,6eb59572-ff45-4ed5-969e-6112b5908899,PREDICTED_WITH,2026-11-04T18:03:18.059513+00:00
4,121e9e2d-db98-432f-a32a-9d34e704e72d,f2088b18-0a06-4413-872b-3573b404aa23,4541467c-6cf9-4ca6-b9b4-3570a8149122,CONTRIBUTED_TO,2026-11-04T18:58:35.755561+00:00
...,...,...,...,...,...
73,f1124729-397e-4077-b48d-769f8417f2ca,4b192779-be1a-46b7-9b9c-d8a1b3f20c35,3c910bc8-5bf7-4818-86d4-2278a9e3ae77,CONTRIBUTED_TO,2026-11-04T19:29:06.312865+00:00
74,fa7c386c-7e4b-441c-8cfa-2ed2aa552cb5,ba351ae2-d087-42ef-bd14-06594cc91f09,9a04a131-a72e-439e-8162-1d156d68df2b,PREDICTED_WITH,2026-11-04T18:13:30.041129+00:00
75,fcb13175-1514-47dc-9da3-0121beaea738,70ae9d3c-2b48-4bd4-a565-91d18ec39d42,dde8f698-7130-47cf-b1a3-f653b683667a,TRAINED_ON,2026-11-04T19:24:42.416011+00:00
76,fcc1fc1c-fc9d-4fad-b078-0181e519f368,bd8c45c2-c137-45e5-92d1-d89be5b1f94b,f52962cd-0110-4425-ae7f-1e28570bf565,CONTRIBUTED_TO,2026-11-04T20:12:06.733851+00:00


2. Grafo e verificações reais. VERIFIED exige presença/proveniência; MISSING/MISMATCH bloqueiam confiança; legado continua identificado.

In [3]:
from IPython.display import HTML
from src.workbench.lineage_review import render_graph
display(HTML(render_graph(evidence)))
display(pd.DataFrame(evidence["checks"]))


snapshot: fb931e1e-43d0-5ff8-938c-0d2ba4a68794,→ TRAINED_ON →,run: f543e6c592254507a10b7e9f1381c94c
inference: 41ee90fe-0f08-48bf-a47a-6503edeac8c7,→ CONTRIBUTED_TO →,proposal: 8ba35d3e-70ee-4719-943a-4b607d2549ee
model: hi.demo.integration.d0b9bf81/1,→ PREDICTED_WITH →,inference: d987f81e-271d-4d0a-90de-4d5eec6812e9
model: planilha-turmas-demo/14,→ PREDICTED_WITH →,inference: 0f45a3cf-0864-4c07-85ac-56280f1882a2
campaign: 28d5c5e0-e21d-458c-8e0a-3aca7b4acc31,→ CONTRIBUTED_TO →,run: 9db2d218bc3845e1841f2f7a21f624a5
inference: 08e86401-59f1-421d-87a5-c43b676652e9,→ CONTRIBUTED_TO →,proposal: b6b8b14b-33a1-4de1-bc01-ca839afdde2d
run: e400ad37fd404c7ab55e7d50b4b16282,→ REGISTERED_AS →,model: hi.demo.probe.168e4337/1
inference: 0b561125-f58b-492b-bd0b-4cdb2f48a800,→ CONTRIBUTED_TO →,proposal: 9c95f92a-87c7-4c86-8922-27fda3942b1d
inference: f743a4cc-f270-4bf2-8adc-131e4be26a08,→ CONTRIBUTED_TO →,proposal: b2ad0068-b2c3-489e-94c0-c69a0b82224e
inference: 5ea3b815-17af-4779-a7b2-11b47da42216,→ CONTRIBUTED_TO →,proposal: 81d0a4af-b0d4-4a3f-b99e-671159d59fed
model: planilha-turmas-demo/10,→ CONTRIBUTED_TO →,proposal: 93519267-7809-4e64-9e04-a79b9fb03d40


,kind,id,status,evidence,url
0,campaign,28d5c5e0-e21d-458c-8e0a-3aca7b4acc31,VERIFIED,PostgreSQL lineage entity,NaN
1,campaign,39f8c7aa-0e6a-463c-aa77-3748f6d118d5,VERIFIED,PostgreSQL lineage entity,NaN
2,campaign,4e413355-a21c-4c13-9311-0320857de96c,VERIFIED,PostgreSQL lineage entity,NaN
3,campaign,5f1ac7f1-ce17-4ec9-bdaf-53f84853b673,VERIFIED,PostgreSQL lineage entity,NaN
4,campaign,7d156255-ee18-4fb6-9c09-a7626dd5e1da,VERIFIED,PostgreSQL lineage entity,NaN
...,...,...,...,...,...
56,run,c78c2971d2d246efb3fcacc5cdf4cd71,VERIFIED,PostgreSQL lineage entity,http://127.0.0.1:5000/#/experiments/4/runs/c78...
57,run,e400ad37fd404c7ab55e7d50b4b16282,VERIFIED,PostgreSQL lineage entity,http://127.0.0.1:5000/#/experiments/4/runs/e40...
58,run,e5eb660aa44b4542b7281105c34613da,VERIFIED,PostgreSQL lineage entity,http://127.0.0.1:5000/#/experiments/4/runs/e5e...
59,run,f543e6c592254507a10b7e9f1381c94c,VERIFIED,PostgreSQL lineage entity,http://127.0.0.1:5000/#/experiments/4/runs/f54...


Guardar evidência desta execução e links. Esperado: caminho do relatório, sem credenciais.

In [4]:
report_status="PARTIAL" if any(isinstance(value,str) and value.startswith("NOT_EXECUTED") for value in evidence.values()) else "PASS"
display(report(NOTEBOOK_ID,evidence,status=report_status,limitations=["Dados DEMO e contrato provisório; regras não avaliadas não são homologação", "Execução sem erro do notebook não implica execução de operações opt-in desativadas"]))


'/reports/v2/08_auditoria_lineage.json'